# Document Summarization


This notebook demonstrates an application of long document summarization techniques to a work of literature.

## Install Dependencies


Install the Python packages used by the notebook. Uncomment and run this cell once if they are not already installed in the current environment.

In [1]:
# %pip install -U transformers langchain-ollama langchain-openrouter python-dotenv docling requests

## Provider setup


When `PROVIDER=ollama`, this notebook uses a locally accessible [Ollama](https://github.com/ollama/ollama) server. Pull the exact quantized model in a terminal before running the notebook:

```powershell
ollama pull LiquidAI/lfm2.5-2.6b:q4_k_m
```

To try it interactively from a terminal, run the model and set `num_gpu` before sending a prompt:

```text
ollama run LiquidAI/lfm2.5-2.6b:q4_k_m
>>> /set parameter num_gpu 0
>>> Summarize the main idea of Walden in one paragraph.
```

The Python client below sends `num_gpu=0` with every request, which forces CPU-only inference. While a request is running, `ollama ps` should show `100% CPU` in the `PROCESSOR` column.

## Select your model

The provider is selected from the project's `.env` file. Set `PROVIDER=ollama` to use the local CPU-only model, or `PROVIDER=openrouter` to use the model named by `OPENROUTER_MODEL`. OpenRouter also requires `OPENROUTER_API_KEY`.

```dotenv
# Local CPU inference
PROVIDER=ollama

# Or hosted inference
# PROVIDER=openrouter
# OPENROUTER_API_KEY=your-key
# OPENROUTER_MODEL=provider/model-name
```

For Ollama, a 32K working context is used by default to limit memory use. The model supports up to 128K, but allocating the full context uses considerably more RAM. You can override it with `OLLAMA_NUM_CTX` in `.env`.

In [2]:
import os

import requests
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from langchain_openrouter import ChatOpenRouter

load_dotenv()

PROVIDER = os.getenv("PROVIDER", "ollama").strip().lower()
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "LiquidAI/lfm2.5-2.6b:q4_k_m")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "").strip()
TOKENIZER_NAME = os.getenv("TOKENIZER_NAME", "LiquidAI/LFM2.5-2.6B")
OLLAMA_BASE_URL = os.getenv("OLLAMA_HOST", "http://127.0.0.1:11434").rstrip("/")
CONTEXT_LENGTH = int(os.getenv("OLLAMA_NUM_CTX", "32768"))
MAX_OUTPUT_TOKENS = int(os.getenv("MAX_OUTPUT_TOKENS", "2048"))

if PROVIDER == "ollama":
    response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=10)
    response.raise_for_status()
    installed_models = {item["name"] for item in response.json().get("models", [])}
    if OLLAMA_MODEL not in installed_models:
        raise RuntimeError(f"{OLLAMA_MODEL} is not installed. Run: ollama pull {OLLAMA_MODEL}")

    model = ChatOllama(
        model=OLLAMA_MODEL,
        base_url=OLLAMA_BASE_URL,
        validate_model_on_init=True,
        reasoning=False,
        num_gpu=0,  # Force CPU-only inference.
        num_ctx=CONTEXT_LENGTH,
        num_predict=MAX_OUTPUT_TOKENS,
        temperature=0.1,
        top_k=50,
        repeat_penalty=1.1,
    )
    MODEL_NAME = OLLAMA_MODEL
elif PROVIDER == "openrouter":
    if not os.getenv("OPENROUTER_API_KEY"):
        raise RuntimeError("OPENROUTER_API_KEY is missing from .env")
    if not OPENROUTER_MODEL:
        raise RuntimeError("OPENROUTER_MODEL is missing from .env")

    model = ChatOpenRouter(
        model=OPENROUTER_MODEL,
        max_tokens=MAX_OUTPUT_TOKENS,
        temperature=1.0,
        top_p=0.95,
    )
    MODEL_NAME = OPENROUTER_MODEL
else:
    raise ValueError("PROVIDER must be either 'ollama' or 'openrouter'")

print(f"Provider: {PROVIDER}; model: {MODEL_NAME}")


d:\Workspace\Repository\llm-summarization\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


Provider: ollama; model: LiquidAI/lfm2.5-2.6b:q4_k_m


In [3]:
# Confirm the configured model can answer before starting summarization.
sanity_response = model.invoke("Hello!")
print(f"Sanity check response: {sanity_response.content}")


Sanity check response: <think>The user said "Hello!" which is a standard greeting. I should respond in a friendly, helpful manner. I'll greet them back and offer assistance.</think>Hello! How can I help you today?


## Split a book into chunks

The code in the following sections will fetch H.D. Thoreau's "Walden" from [Project Gutenberg](https://www.gutenberg.org/) for summarization.

We will then chunk the book text so that chunks fit in the context window size of the AI model.

### Count the tokens

Before sending our book chunks to the AI model, it's crucial to understand how much of the model's capacity we're using. Language models typically have a limit on the number of tokens they can process in a single request.

Key points:
- [`LFM2.5-2.6B`](https://huggingface.co/LiquidAI/LFM2.5-2.6B) supports a context window of up to 128K tokens.
- Ollama uses the smaller working context configured above to reduce CPU RAM use.
- Tokenization varies between models. `TOKENIZER_NAME` defaults to LFM2.5-2.6B's tokenizer and can be overridden in `.env` when selecting a different OpenRouter model.

Understanding token count helps us optimize our prompts and ensure we're using the model efficiently.

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, trust_remote_code=True)

### Summary of Summaries

Here we use a hierarchical abstractive summarization technique to adapt to the context length of the model. Our approach uses [Docling](https://docling-project.github.io/docling/) to understand the document's structure, chunk the document into text passages, and group the text passages by chapter which we can then summarize. 

In [5]:
import itertools
from collections.abc import Callable, Iterator

from docling.document_converter import DocumentConverter
from docling_core.transforms.chunker.base import BaseChunk
from docling_core.transforms.chunker.hierarchical_chunker import HierarchicalChunker


def chunk_document(source: str, *, dropwhile: Callable[[BaseChunk], bool] = lambda c: False, takewhile: Callable[[BaseChunk], bool] = lambda c: True) -> Iterator[BaseChunk]:
    """Read the document and perform a hierarchical chunking"""
    converter = DocumentConverter()
    chunks = HierarchicalChunker().chunk(converter.convert(source=source).document)
    return itertools.takewhile(takewhile, itertools.dropwhile(dropwhile, chunks))

def merge_chunks(chunks: Iterator[BaseChunk], *, headings: Callable[[BaseChunk], list[str]] = lambda c: c.meta.headings) -> Iterator[dict[str, str]]:
    """Merge chunks having the same headings"""
    prior_headings: list[str] | None = None
    document: dict[str, str] = {}
    doc_id = 0
    for chunk in chunks:
        text = chunk.text.replace('\r\n', '\n')
        current_headings = headings(chunk)
        if prior_headings != current_headings:
            if document:
                yield document
            prior_headings = current_headings
            document = {
                'doc_id': str(doc_id := doc_id + 1),
                'title': " - ".join(current_headings),
                'text': text
            }
        else:
            document['text'] += f"\n\n{text}"
    if document:
        yield document

def chunk_dropwhile(chunk: BaseChunk) -> bool:
    """Ignore front matter prior to the book start"""
    return chunk.meta.headings != ["WALDEN", "Economy"]

def chunk_takewhile(chunk: BaseChunk) -> bool:
    """Ignore remaining chunks once we see this heading"""
    return "ON THE DUTY OF CIVIL DISOBEDIENCE" not in chunk.meta.headings

def chunk_headings(chunk: BaseChunk) -> list[str]:
    """Use the h1 and h2 (chapter) headings"""
    return chunk.meta.headings[:2]

documents: list[dict[str, str]] = list(merge_chunks(
    chunk_document(
        "https://www.gutenberg.org/cache/epub/205/pg205-images.html",
        dropwhile=chunk_dropwhile,
        takewhile=chunk_takewhile,
    ),
    headings=chunk_headings,
))

print(f"{len(documents)} documents created")
print(f"Max document size: {max(len(tokenizer.tokenize(document['text'])) for document in documents)} tokens")

18 documents created
Max document size: 32043 tokens


## Summarize the chunks

Here we define methods to generate a response, separate Ollama-style `<think>` blocks from the final answer, and display the result clearly. Each model call also records its input tokens, output tokens, and elapsed wall-clock time.

The documents are placed directly in standard chat messages so either provider can apply the selected model's native chat template.

In [6]:
import re
from html import escape
from time import perf_counter

from IPython.display import HTML, display
from langchain_core.messages import HumanMessage, SystemMessage

SYSTEM_PROMPT = "You are a careful document summarizer. Use only the supplied source documents."
THINKING_BLOCK_PATTERN = re.compile(r"<think>(.*?)</think>", re.IGNORECASE | re.DOTALL)
DOCUMENT_TAG_PATTERN = re.compile(r"</?document(?:\s+[^>]*)?>", re.IGNORECASE)


def split_llm_output(text: str) -> tuple[str, str]:
    """Separate Ollama-style thinking blocks from the final answer."""
    thinking_sections = [
        match.strip()
        for match in THINKING_BLOCK_PATTERN.findall(text)
        if match.strip()
    ]
    final_answer = THINKING_BLOCK_PATTERN.sub("", text)
    final_answer = DOCUMENT_TAG_PATTERN.sub("", final_answer).strip()
    return "\n\n".join(thinking_sections), final_answer


def display_llm_output(text: str) -> str:
    """Display reasoning in a collapsed panel and the final answer separately."""
    thinking, final_answer = split_llm_output(text)
    sections: list[str] = []

    if thinking:
        sections.append(
            "<details style='margin: 0 0 1rem; padding: 0.75rem 1rem; "
            "border: 1px solid var(--jp-border-color2, #d0d7de); border-radius: 6px;'>"
            "<summary style='cursor: pointer; font-weight: 600;'>"
            "Thinking (click to expand)</summary>"
            "<div style='margin-top: 0.75rem; white-space: pre-wrap; "
            "line-height: 1.5; opacity: 0.8;'>"
            f"{escape(thinking)}</div></details>"
        )

    answer_to_display = final_answer or "(No final answer was returned.)"
    sections.append(
        "<h3 style='margin-bottom: 0.5rem;'>Final answer</h3>"
        "<div style='white-space: pre-wrap; line-height: 1.6;'>"
        f"{escape(answer_to_display)}</div>"
    )
    display(HTML("".join(sections)))
    return final_answer


def generate(user_prompt: str, documents: list[dict[str, str]]) -> tuple[str, dict[str, float | int | str]]:
    """Invoke the model and return its text plus timing and token measurements."""
    document_text = "\n\n".join(
        f"<document id='{document['doc_id']}' title='{document['title']}'>\n{document['text']}\n</document>"
        for document in documents
    )
    request_text = f"{user_prompt}\n\nSource documents:\n{document_text}"
    chat = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": request_text},
    ]
    formatted_chat = tokenizer.apply_chat_template(
        chat, tokenize=False, add_generation_prompt=True
    )
    estimated_input_tokens = len(tokenizer.encode(
        formatted_chat, add_special_tokens=False
    ))
    if PROVIDER == "ollama":
        available_output_tokens = CONTEXT_LENGTH - estimated_input_tokens
        if available_output_tokens <= 0:
            raise ValueError(
                f"The input needs about {estimated_input_tokens:,} tokens, but "
                f"OLLAMA_NUM_CTX is {CONTEXT_LENGTH:,}. Increase it or use smaller documents."
            )
        if available_output_tokens < MAX_OUTPUT_TOKENS:
            print(
                f"Warning: only about {available_output_tokens:,} context tokens remain "
                f"for output, below MAX_OUTPUT_TOKENS={MAX_OUTPUT_TOKENS:,}."
            )

    print(f"Estimated input size: {estimated_input_tokens:,} tokens")
    started_at = perf_counter()
    response = model.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=request_text),
    ])
    elapsed_seconds = perf_counter() - started_at
    output_text = response.content if isinstance(response.content, str) else str(response.content)
    usage = response.usage_metadata or {}
    provider_input_tokens = usage.get("input_tokens")
    provider_output_tokens = usage.get("output_tokens")
    input_tokens = (
        provider_input_tokens
        if provider_input_tokens is not None
        else estimated_input_tokens
    )
    output_tokens = (
        provider_output_tokens
        if provider_output_tokens is not None
        else len(tokenizer.encode(output_text, add_special_tokens=False))
    )
    token_count_source = (
        "provider"
        if provider_input_tokens is not None and provider_output_tokens is not None
        else "tokenizer estimate"
    )
    total_tokens = input_tokens + output_tokens
    tokens_per_second = total_tokens / elapsed_seconds if elapsed_seconds else float("inf")

    if provider_input_tokens is not None:
        print(f"Provider-reported input size: {provider_input_tokens:,} tokens")
    print(f"Output size: {output_tokens:,} tokens")
    print(f"Elapsed: {elapsed_seconds:,.2f} seconds; throughput: {tokens_per_second:,.2f} tokens/s")

    metrics = {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "elapsed_seconds": elapsed_seconds,
        "tokens_per_second": tokens_per_second,
        "token_count_source": token_count_source,
    }
    return output_text, metrics

For each chapter, we create a separate summary. CPU-only inference can take a while, especially for the longer chapters. Set `SUMMARIZATION_TESTING=true` before starting Jupyter to process only the first five chapters. The source-text measurements below describe exactly the chapters included in the current run.

In [7]:
if os.getenv("SUMMARIZATION_TESTING", "false").lower() == "true":
    print("SUMMARIZATION_TESTING is true, only processing the first 5 documents")
    documents = documents[:5]  # Shorten testing work.

source_text = "\n\n".join(document["text"] for document in documents)
source_characters = len(source_text)
source_words = len(source_text.split())
source_tokens = len(tokenizer.encode(source_text, add_special_tokens=False))
summarization_scope = f"{len(documents)} chapter(s)"

print(
    f"Source text for this run: {source_characters:,} characters, "
    f"{source_words:,} words, {source_tokens:,} tokens"
)

user_prompt = """\
Using only the book chapter document, compose a summary of the book chapter.
Your response should only include the summary. Do not provide any further explanation."""

summaries: list[dict[str, str]] = []
run_metrics: list[dict[str, float | int | str]] = []
summarization_started_at = perf_counter()

for i, document in enumerate(documents, start=1):
    print(f"============================= {document['title']} ({i}/{len(documents)}) =============================")
    output, call_metrics = generate(user_prompt, [document])
    _, summary_text = split_llm_output(output)
    run_metrics.append({
        "stage": "chapter",
        "title": document["title"],
        **call_metrics,
    })
    summaries.append({
        'doc_id': document['doc_id'],
        'title': document['title'],
        'text': summary_text or output.strip(),
    })

print("Summary count: " + str(len(summaries)))

Source text for this run: 581,844 characters, 106,505 words, 134,019 tokens
============================= WALDEN - Economy (1/18) =============================
Estimated input size: 32,123 tokens
Provider-reported input size: 32,122 tokens
Output size: 840 tokens
Elapsed: 412.77 seconds; throughput: 79.86 tokens/s
============================= WALDEN - Where I Lived, and What I Lived For (2/18) =============================
Estimated input size: 7,671 tokens
Provider-reported input size: 7,670 tokens
Output size: 627 tokens
Elapsed: 85.56 seconds; throughput: 96.97 tokens/s
============================= WALDEN - Reading (3/18) =============================
Estimated input size: 4,699 tokens
Provider-reported input size: 4,698 tokens
Output size: 737 tokens
Elapsed: 63.82 seconds; throughput: 85.16 tokens/s
============================= WALDEN - Sounds (4/18) =============================
Estimated input size: 7,482 tokens
Provider-reported input size: 7,481 tokens
Output size: 633 toke

## Create the Final Summary

Now we need to summarize the chapter summaries. We prompt the model to create a unified summary of the chapter summaries we previously generated.

In [8]:
user_prompt = """\
Using only the book chapter summary documents, compose a single, unified summary of the book.
Your response should only include the unified summary. Do not provide any further explanation."""

output, final_call_metrics = generate(user_prompt, summaries)
run_metrics.append({
    "stage": "final synthesis",
    "title": "Walden",
    **final_call_metrics,
})
summarization_elapsed_seconds = perf_counter() - summarization_started_at

total_input_tokens = sum(int(item["input_tokens"]) for item in run_metrics)
total_output_tokens = sum(int(item["output_tokens"]) for item in run_metrics)
total_model_tokens = total_input_tokens + total_output_tokens
model_call_seconds = sum(float(item["elapsed_seconds"]) for item in run_metrics)
overall_tokens_per_second = (
    total_model_tokens / summarization_elapsed_seconds
    if summarization_elapsed_seconds
    else float("inf")
)
pipeline_token_multiplier = (
    total_model_tokens / source_tokens if source_tokens else float("inf")
)
token_count_source = (
    "provider-reported"
    if all(item["token_count_source"] == "provider" for item in run_metrics)
    else "provider-reported where available, otherwise tokenizer estimates"
)

display_llm_output(output)
print("\n==================== RUN METRICS ====================")
print(f"Summarization scope: {summarization_scope}")
print(f"Source text processed: {source_characters:,} characters")
print(f"Source words processed: {source_words:,}")
print(f"Source tokens: {source_tokens:,}")
print(f"Model calls: {len(run_metrics):,}")
print(f"Cumulative model input tokens: {total_input_tokens:,}")
print(f"Cumulative model output tokens: {total_output_tokens:,}")
print(f"Total model tokens processed: {total_model_tokens:,}")
print(f"Pipeline token multiplier: {pipeline_token_multiplier:,.2f}x source tokens")
print(f"Model-call time: {model_call_seconds:,.2f} seconds")
print(
    f"Total summarization time: {summarization_elapsed_seconds:,.2f} seconds "
    f"({summarization_elapsed_seconds / 60:,.2f} minutes)"
)
print(f"End-to-end throughput: {overall_tokens_per_second:,.2f} tokens/s")
print(f"Token counts: {token_count_source}")

Estimated input size: 6,068 tokens
Provider-reported input size: 6,067 tokens
Output size: 3,921 tokens
Elapsed: 239.64 seconds; throughput: 41.68 tokens/s



==================== RUN METRICS ====================
Summarization scope: 18 chapter(s)
Source text processed: 581,844 characters
Source words processed: 106,505
Source tokens: 134,019
Model calls: 19
Cumulative model input tokens: 141,525
Cumulative model output tokens: 16,209
Total model tokens processed: 157,734
Pipeline token multiplier: 1.18x source tokens
Model-call time: 1,876.07 seconds
Total summarization time: 1,877.78 seconds (31.30 minutes)
End-to-end throughput: 84.00 tokens/s
Token counts: provider-reported


So we have now summarized a document larger than the AI model's context window length by breaking the document down into smaller pieces to summarize and then summarizing those summaries.

The final report separates the unique source size from cumulative model traffic. Cumulative input is larger because every request includes instructions and markup, and the final request reads all chapter summaries. `End-to-end throughput` is total input plus output tokens divided by wall-clock summarization time. It includes request overhead, so it is useful for estimating another complete document run. Document download, parsing, and chunking occur before the timer and are not included.

For a rough estimate with the same model and settings: `estimated model tokens = new source tokens × pipeline token multiplier`, then `estimated seconds = estimated model tokens ÷ end-to-end throughput`. This scales approximately, but chapter sizes, summary lengths, hardware load, and hosted-provider latency can change the result.